# Эмбеддинги корпусов на Colab (EXP-019)

Считает `embed_corpus` для `split` и `benchmark` на GPU Colab. Вход и выход лежат на Google Drive в `MyDrive/avito/`:

- `src.zip` — пакет `candgen` (`src/`);
- `corpus.parquet` — `data/artifacts/split/corpus.parquet`;
- `benchmark_items.parquet` — `data/input/benchmark_items.parquet`.

Результат: `MyDrive/avito/embeddings/<corpus>_<model>_len512_p500/`. Чанки по 50 000 пишутся сразу на Drive, поэтому после обрыва сессии повторный запуск продолжает с последнего чанка.

In [1]:
MODEL = "deepvk/USER-base"
CORPORA = ["split", "benchmark"]
BATCH_SIZE = 64
DRIVE = "/content/drive/MyDrive/avito"
WORK = "/content/avito"

In [2]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [3]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
!pip -q install "sentence-transformers==6.0.1" "transformers==5.17.0" "polars==1.44.2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 20.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 122.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.8/865.8 kB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 21.2 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-polars-cu12 26.2.1 requires polars<1.36,>=1.30, but you have polars 1.44.2 which is incompatible.


In [5]:
import os
from pathlib import Path

work = Path(WORK)
(work / "data/artifacts/split").mkdir(parents=True, exist_ok=True)
(work / "data/input").mkdir(parents=True, exist_ok=True)
(Path(DRIVE) / "embeddings").mkdir(parents=True, exist_ok=True)
!cd {WORK} && unzip -oq {DRIVE}/src.zip
!cp -n {DRIVE}/corpus.parquet {WORK}/data/artifacts/split/
!cp -n {DRIVE}/benchmark_items.parquet {WORK}/data/input/
link = work / "data/artifacts/embeddings"
if not link.exists():
    link.symlink_to(Path(DRIVE) / "embeddings")
os.chdir(work)
!ls -la data/artifacts/split data/input data/artifacts/embeddings

cp: warning: behavior of -n is non-portable and may change in future; use --update=none instead
cp: warning: behavior of -n is non-portable and may change in future; use --update=none instead
lrwxrwxrwx 1 root root   39 Sep 19 10:04 data/artifacts/embeddings -> /content/drive/MyDrive/avito/embeddings

data/artifacts/split:
total 286628
drwxr-xr-x 2 root root      4096 Sep 19 10:04 .
drwxr-xr-x 3 root root      4096 Sep 19 10:04 ..
-rw------- 1 root root 293490695 Sep 19 10:04 corpus.parquet

data/input:
total 191108
drwxr-xr-x 2 root root      4096 Sep 19 10:04 .
drwxr-xr-x 4 root root      4096 Sep 19 10:04 ..
-rw------- 1 root root 195679553 Sep 19 10:04 benchmark_items.parquet


In [ ]:
for corpus in CORPORA:
    !PYTHONPATH=src python -m candgen.scripts.embed_corpus --corpus {corpus} --model {MODEL} --batch-size {BATCH_SIZE}

modules.json: 100% 338/338 [00:00<00:00, 1.39MB/s]
config_sentence_transformers.json: 100% 232/232 [00:00<00:00, 1.12MB/s]
README.md: 100% 12.1k/12.1k [00:00<00:00, 2.48MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 171kB/s]
config.json: 100% 700/700 [00:00<00:00, 2.58MB/s]

model.safetensors: downloading bytes:  43% 213M/496M [00:01<00:01, 257MB/s, 16.1MB/s  ]  
model.safetensors: downloading bytes:  60% 299M/496M [00:01<00:00, 225MB/s, 25.5MB/s  ]
model.safetensors: downloading bytes:  85% 421M/496M [00:02<00:00, 284MB/s, 34.4MB/s  ] ]
model.safetensors: reconstructing file:  68% 335M/496M [00:02<00:01, 154MB/s, 24.9MB/s  ]
model.safetensors: downloading bytes:  92% 458M/496M [00:02<00:00, 141MB/s, 39.0MB/s  ] ]
model.safetensors: downloading bytes: 100% 470M/470M [00:03<00:00, 142MB/s, 39.2MB/s  ] ]
model.safetensors: reconstructing file: 100% 496M/496M [00:03<00:00, 150MB/s, 43.1MB/s  ]
Loading weights: 100% 160/160 [00:00<00:00, 8028.91it/s]
tokenizer_config.json: 1

In [ ]:
!ls -la {DRIVE}/embeddings/*
!cat {DRIVE}/embeddings/*/meta.json | grep -E "corpus|encode_s|truncated_share|dtype"